In [1]:
!rm -rf diploma_centpy_parallelization_py
# Флаг -b указывает конкретную ветку
!git clone -b feature/jax-centpy https://github.com/filkinc/diploma_centpy_parallelization_py.git
%cd diploma_centpy_parallelization_py
%cd /content/diploma_centpy_parallelization_py/jax_centpy

# Установка зависимостей
!pip install centpy pandas matplotlib seaborn

Cloning into 'diploma_centpy_parallelization_py'...
remote: Enumerating objects: 191, done.
remote: Counting objects: 100% (191/191), done.
remote: Compressing objects: 100% (142/142), done.
remote: Total 191 (delta 75), reused 161 (delta 47), pack-reused 0 (from 0)
Receiving objects: 100% (191/191), 20.28 MiB | 17.46 MiB/s, done.
Resolving deltas: 100% (75/75), done.
/content/diploma_centpy_parallelization_py
/content/diploma_centpy_parallelization_py/jax_centpy


In [8]:
import os
import time
import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import numpy as np

from core import Pars2d, Equation2d
from solver import Solver2d, FastSolver2d
from boundaries import periodic_bc_2d
from equations import make_euler_riemann_2d
from schemes import compute_rhs_sd2_2d
from limiters import monotonized_central, minmod
from boundaries import neumann_bc_2d

In [10]:
# Фиксируем ту же сетку 6x6
pars_jax = Pars2d(x_init=0.0, x_final=1.0, y_init=0.0, y_final=1.0,
                  t_final=0.1, dt_out=0.1, Jx=6, Jy=6, cfl=0.475, scheme="sd2")
eqn_jax = make_euler_riemann_2d(gamma=1.4)

# Генерируем сетку так, как это делает JAX-солвер
x1d = jnp.linspace(pars_jax.x_init + pars_jax.dx / 2, pars_jax.x_final - pars_jax.dx / 2, pars_jax.Jx)
y1d = jnp.linspace(pars_jax.y_init + pars_jax.dy / 2, pars_jax.y_final - pars_jax.dy / 2, pars_jax.Jy)
X, Y = jnp.meshgrid(x1d, y1d, indexing="ij")

u_init_jax = eqn_jax.initial_data(X, Y)
dt = 0.001

# В JAX-версии массив u хранит только внутренние ячейки (без ghost), размер 6x6.
# Делаем один шаг RK2 (абсолютно то же самое, что внутри метода sd2 в centpy)
rhs = compute_rhs_sd2_2d(0.0, u_init_jax, pars_jax, eqn_jax, limiter=minmod, theta=1.0)
u1_star = u_init_jax + dt * rhs
rhs_star = compute_rhs_sd2_2d(0.0, u1_star, pars_jax, eqn_jax, limiter=minmod, theta=1.0)
u_step1_jax = 0.5 * u_init_jax + 0.5 * (u1_star + dt * rhs_star)

mid = 6 // 2
print("--- GPU (JAX) ---")
print("u_init (плотность rho) в центре 2x2:")
print(u_init_jax[mid-1:mid+1, mid-1:mid+1, 0])
print("\nu_step1 (плотность rho) в центре 2x2:")
print(u_step1_jax[mid-1:mid+1, mid-1:mid+1, 0])

--- GPU (JAX) ---
u_init (плотность rho) в центре 2x2:
[[0.138  0.5323]
 [0.5323 1.5   ]]

u_step1 (плотность rho) в центре 2x2:
[[0.14313807 0.53863027]
 [0.53863027 1.49187201]]


In [12]:
def make_euler_riemann_2d_fixed(gamma: float = 1.4):

    def _compute_pressure(q):
        rho, rhou, rhov, E = q[..., 0], q[..., 1], q[..., 2], q[..., 3]
        u = rhou / rho
        v = rhov / rho
        return (gamma - 1.0) * (E - 0.5 * rho * (u ** 2 + v ** 2))

    def flux_x(q):
        rho, rhou, rhov, E = q[..., 0], q[..., 1], q[..., 2], q[..., 3]
        u = rhou / rho
        p = _compute_pressure(q)
        return jnp.stack([rhou, rhou * u + p, rhou * (rhov / rho), u * (E + p)], axis=-1)

    def flux_y(q):
        rho, rhou, rhov, E = q[..., 0], q[..., 1], q[..., 2], q[..., 3]
        v = rhov / rho
        p = _compute_pressure(q)
        return jnp.stack([rhov, rhou * v, rhov * v + p, v * (E + p)], axis=-1)

    def spectral_radius_x(q):
        rho, rhou = q[..., 0], q[..., 1]
        u = rhou / rho
        # Защита как в CPU-версии
        p = jnp.maximum(_compute_pressure(q), 1e-10)
        rho = jnp.maximum(rho, 1e-10)
        return jnp.abs(u) + jnp.sqrt(gamma * p / rho)

    def spectral_radius_y(q):
        rho, rhov = q[..., 0], q[..., 2]
        v = rhov / rho
        p = jnp.maximum(_compute_pressure(q), 1e-10)
        rho = jnp.maximum(rho, 1e-10)
        return jnp.abs(v) + jnp.sqrt(gamma * p / rho)

    def initial_riemann(x, y):
        rho = jnp.where((x > 0.5) & (y > 0.5), 1.5,
                        jnp.where((x <= 0.5) & (y > 0.5), 0.5323,
                                  jnp.where((x <= 0.5) & (y <= 0.5), 0.138, 0.5323)))
        vx = jnp.where((x > 0.5) & (y > 0.5), 0.0,
                       jnp.where((x <= 0.5) & (y > 0.5), 1.206,
                                 jnp.where((x <= 0.5) & (y <= 0.5), 1.206, 0.0)))
        vy = jnp.where((x > 0.5) & (y > 0.5), 0.0,
                       jnp.where((x <= 0.5) & (y > 0.5), 0.0,
                                 jnp.where((x <= 0.5) & (y <= 0.5), 1.206, 1.206)))
        p = jnp.where((x > 0.5) & (y > 0.5), 1.5,
                      jnp.where((x <= 0.5) & (y > 0.5), 0.3,
                                jnp.where((x <= 0.5) & (y <= 0.5), 0.029, 0.3)))

        E = p / (gamma - 1.0) + 0.5 * rho * (vx ** 2 + vy ** 2)
        return jnp.stack([rho, rho * vx, rho * vy, E], axis=-1)

    # === НАШИ НОВЫЕ ГРАНИЧНЫЕ УСЛОВИЯ ===
    def boundary_conditions_jax(u_inner, n_ghost=2):
        # 1. Экстраполяция Неймана (копирует внутренние крайние ячейки в ghost-зону)
        # Это позволяет волнам "вытекать" без отражений. mode='edge' делает ровно то же, что u[j, 0] = u[j, 2]
        u_pad = jnp.pad(u_inner, ((n_ghost, n_ghost), (n_ghost, n_ghost), (0, 0)), mode='edge')

        # 2. Фиксируем углы на начальных значениях квадрантов (как в твоем CPU коде)
        p_one, p_two, p_three, p_four = 1.5, 0.3, 0.029, 0.3

        ur = jnp.array([1.5, 0.0, 0.0, p_one / (gamma - 1.0)], dtype=u_inner.dtype)

        ul_0 = 0.5323
        ul_1 = 1.206 * ul_0
        ul = jnp.array([ul_0, ul_1, 0.0, p_two / (gamma - 1.0) + 0.5 * ul_1**2 / ul_0], dtype=u_inner.dtype)

        lr_0 = 0.5323
        lr_2 = 1.206 * lr_0
        lr = jnp.array([lr_0, 0.0, lr_2, p_four / (gamma - 1.0) + 0.5 * lr_2**2 / lr_0], dtype=u_inner.dtype)

        ll_0 = 0.138
        ll_1 = 1.206 * ll_0
        ll_2 = 1.206 * ll_0
        ll = jnp.array([ll_0, ll_1, ll_2, p_three / (gamma - 1.0) + 0.5 * (ll_1**2 + ll_2**2) / ll_0], dtype=u_inner.dtype)

        # Перезаписываем угловые блоки (размером n_ghost x n_ghost)
        u_pad = u_pad.at[-n_ghost:, -n_ghost:].set(ur) # Верхний правый
        u_pad = u_pad.at[:n_ghost, -n_ghost:].set(ul)  # Верхний левый
        u_pad = u_pad.at[-n_ghost:, :n_ghost].set(lr)  # Нижний правый
        u_pad = u_pad.at[:n_ghost, :n_ghost].set(ll)   # Нижний левый

        return u_pad

    return Equation2d(
        flux_x=flux_x, flux_y=flux_y,
        spectral_radius_x=spectral_radius_x, spectral_radius_y=spectral_radius_y,
        initial_data=initial_riemann,
        boundary_handler=boundary_conditions_jax,
        name="Euler 2D (Riemann)"
    )

def run_gpu_benchmark():
    print(f"JAX Device(s): {jax.devices()}")
    J = 200

    pars = Pars2d(
        x_init=0.0, x_final=1.0, y_init=0.0, y_final=1.0,
        t_final=0.4, dt_out=0.005, Jx=J, Jy=J, cfl=0.475, scheme="sd2"
    )

    eqn = make_euler_riemann_2d_fixed()
    solver = FastSolver2d(pars, eqn, limiter_name="minmod")
    solverForPlot = Solver2d(pars, eqn, scheme_name="sd2", limiter_name="minmod")

    print("--- Прогрев JAX (Компиляция XLA) ---")
    # Прогреваем ТОЛЬКО первый шаг решателя (именно он содержит всю тяжелую математику)
    x_1d = jnp.linspace(pars.x_init + pars.dx / 2, pars.x_final - pars.dx / 2, pars.Jx)
    y_1d = jnp.linspace(pars.y_init + pars.dy / 2, pars.y_final - pars.dy / 2, pars.Jy)
    X, Y = jnp.meshgrid(x_1d, y_1d, indexing='ij')
    test_u = eqn.initial_data(X, Y)
    _ = solver.update_step_jit(0.0, test_u, 0.001).block_until_ready()
    print("Прогрев завершен.\n")

    print(f"--- Запуск JAX GPU Бенчмарка (Euler 2D Riemann, {J}x{J}) ---")
    t0 = time.time()
    results = solver.solve()
    results['u'][-1].block_until_ready()
    t1 = time.time()

    print(f"\n[GPU JAX] Чистое время выполнения: {t1 - t0:.4f} секунд")

    resultsForPlot = solverForPlot.solve()

    return resultsForPlot, pars

if __name__ == "__main__":
    jax.config.update("jax_enable_x64", True)
    soln, pars = run_gpu_benchmark()

JAX Device(s): [CudaDevice(id=0)]
--- Прогрев JAX (Компиляция XLA) ---
Прогрев завершен.

--- Запуск JAX GPU Бенчмарка (Euler 2D Riemann, 200x200) ---
Starting 2D simulation: Euler 2D (Riemann)
Grid: 200x200, Scheme: SD2/minmod

[GPU JAX] Чистое время выполнения: 1.5518 секунд
Starting 2D simulation: Euler 2D (Riemann)
Grid: 200x200, Scheme: SD2/minmod


In [13]:
u_data = soln['u']
X_grid = soln['X']
Y_grid = soln['Y']

# Инициализация фигуры и осей, берем границы из объекта pars
fig, ax = plt.subplots()
ax.set_xlim(pars.x_init, pars.x_final)
ax.set_ylim(pars.y_init, pars.y_final)

# Для первого кадра (плотность, так как индекс [..., 0] соответствует плотности в уравнениях Эйлера)
data_init = u_data[0, ..., 0]

# 1. Создаем фоновую тепловую карту с помощью imshow
im = ax.imshow(
    data_init.T, # Транспонируем, так как imshow ожидает порядок (y, x), а у вас индексация 'ij'
    extent=[pars.x_init, pars.x_final, pars.y_init, pars.y_final],
    origin='lower',
    cmap='coolwarm',            # Золотой стандарт для волновых процессов
    interpolation='bicubic',
    aspect='auto'
)

cbar = fig.colorbar(im, ax=ax)
cbar.set_label('Плотность (u[..., 0])')

# 2. Отрисовываем начальные контуры поверх тепловой карты
ax.contour(
    X_grid, Y_grid, data_init,
    levels=20,
    colors='black',
    alpha=0.5,
    linewidths=0.5
)

# Функция обновления для каждого кадра анимации
def animate(i):
    # Получаем данные текущего шага
    data = u_data[i, ..., 0]

    # Обновляем данные на тепловой карте
    im.set_data(data.T)

    # Динамически обновляем границы цветовой шкалы (опционально)
    im.set_clim(vmin=data.min(), vmax=data.max())

    # Удаляем старые линии контуров из коллекции осей
    for c in ax.collections:
        c.remove()

    # Рисуем новые контурные линии
    ax.contour(
        X_grid, Y_grid, data,
        levels=20,
        colors='black',
        alpha=0.5,
        linewidths=0.5
    )

    return [im]

plt.close() # Закрываем статичную фигуру

# Создаем анимацию (количество кадров равно размеру массива времени)
num_frames = u_data.shape[0]
anim = animation.FuncAnimation(fig, animate, frames=num_frames, interval=100, blit=False)

# Выводим как HTML5 видео
HTML(anim.to_html5_video())